# Binance Spot USDT quote-pair trade counts

This notebook lists every Binance Spot symbol currently returned by `exchangeInfo` whose quote asset is USDT, without filtering by current status. For each symbol, it sums the `number_of_trades` field from daily klines between 2017-01-01 and the end of the previous UTC calendar day. No API key is required.

The count belongs to the complete trading symbol (for example, `BTCUSDT`), not to the base asset independently of its quote asset. `BTCUSDT` and `BTCBRL` have separate order books, trades, and kline counts.

## Historical-universe limitation

Binance describes `exchangeInfo` as **current** exchange information. Not filtering its response retains statuses such as `TRADING`, `BREAK`, and `HALT` when present, but it does not prove that symbols already removed from the current response are included. Therefore, the output is complete for USDT-quoted symbols discoverable through `exchangeInfo` at collection time, not necessarily for every delisted symbol that ever existed.

In [1]:
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd

BASE_URL = "https://api.binance.com"
EXCHANGE_INFO_PATH = "/api/v3/exchangeInfo"
KLINES_PATH = "/api/v3/klines"
QUOTE_ASSET = "USDT"
KLINE_INTERVAL = "1d"
START_TIME_UTC = pd.Timestamp("2017-01-01", tz="UTC")
COLLECTION_TIME_UTC = pd.Timestamp.now(tz="UTC")
END_TIME_UTC = COLLECTION_TIME_UTC.normalize() - pd.Timedelta(milliseconds=1)
START_TIME_MS = int(START_TIME_UTC.timestamp() * 1_000)
END_TIME_MS = int(END_TIME_UTC.timestamp() * 1_000)
KLINE_LIMIT = 1000
REQUEST_TIMEOUT_SECONDS = 30
MAX_RETRIES = 6
MAX_WORKERS = 8


def find_repository_root(start: Path | None = None) -> Path:
    """Find the repository root by looking for AGENTS.md."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")


REPOSITORY_ROOT = find_repository_root()
OUTPUT_PATH = (
    REPOSITORY_ROOT
    / "data"
    / "exploration"
    / "binance_spot_usdt_quote_pair_trade_counts.xlsx"
)
print(f"Collection cutoff (UTC): {END_TIME_UTC.isoformat()}")
OUTPUT_PATH

Collection cutoff (UTC): 2026-09-01T23:59:59.999000+00:00


PosixPath('/Users/marcelapsilverio/projetos/tcc_crypto/data/exploration/binance_spot_usdt_quote_pair_trade_counts.xlsx')

In [2]:
def request_json(
    path: str,
    parameters: dict[str, object] | None = None,
) -> object:
    """Request JSON from Binance with bounded transient-error retries."""
    query_string = urlencode(parameters or {})
    url = f"{BASE_URL}{path}"
    if query_string:
        url = f"{url}?{query_string}"

    for attempt in range(MAX_RETRIES + 1):
        request = Request(
            url, headers={"User-Agent": "tcc-crypto-research/0.1"}
        )
        try:
            with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
                return json.load(response)
        except HTTPError as error:
            is_transient = error.code == 429 or 500 <= error.code < 600
            if is_transient and attempt < MAX_RETRIES:
                retry_after = error.headers.get("Retry-After")
                delay_seconds = (
                    float(retry_after)
                    if retry_after is not None
                    else min(2**attempt, 30)
                )
                time.sleep(delay_seconds)
                continue
            response_body = error.read().decode("utf-8", errors="replace")
            raise RuntimeError(
                f"Binance returned HTTP {error.code} for {url}: {response_body}"
            ) from error
        except URLError as error:
            if attempt < MAX_RETRIES:
                time.sleep(min(2**attempt, 30))
                continue
            raise RuntimeError(
                f"Could not reach Binance at {url}: {error.reason}"
            ) from error

    raise RuntimeError(f"Request retry loop ended unexpectedly for {url}.")

In [3]:
def get_usdt_quote_pairs() -> pd.DataFrame:
    """Return all USDT-quoted symbols in the current Spot catalog."""
    payload = request_json(EXCHANGE_INFO_PATH)
    if not isinstance(payload, dict) or not isinstance(payload.get("symbols"), list):
        raise ValueError("Unexpected Binance exchangeInfo response.")

    records = [
        {
            "Trading Pair": item["symbol"],
            "Base Asset": item["baseAsset"],
            "Quote Asset": item["quoteAsset"],
            "Current Status": item["status"],
            "Spot Trading Currently Allowed": item.get(
                "isSpotTradingAllowed"
            ),
        }
        for item in payload["symbols"]
        if item.get("quoteAsset") == QUOTE_ASSET
    ]
    if not records:
        raise ValueError(f"No symbols quoted in {QUOTE_ASSET} were returned.")
    return pd.DataFrame(records).sort_values("Trading Pair").reset_index(drop=True)


pairs = get_usdt_quote_pairs()
print(f"USDT-quoted symbols returned: {len(pairs):,}")
display(pairs["Current Status"].value_counts(dropna=False).rename("Symbols"))
pairs.head()

USDT-quoted symbols returned: 738


Current Status
TRADING    489
BREAK      249
Name: Symbols, dtype: int64

,Trading Pair,Base Asset,Quote Asset,Current Status,Spot Trading Currently Allowed
0,0GUSDT,0G,USDT,TRADING,True
1,1000CATUSDT,1000CAT,USDT,TRADING,True
2,1000CHEEMSUSDT,1000CHEEMS,USDT,TRADING,True
3,1000SATSUSDT,1000SATS,USDT,TRADING,True
4,1INCHDOWNUSDT,1INCHDOWN,USDT,BREAK,False


In [4]:
KLINE_COLUMNS = [
    "open_time_ms", "open", "high", "low", "close",
    "base_volume", "close_time_ms", "quote_volume",
    "number_of_trades", "taker_buy_base_volume",
    "taker_buy_quote_volume", "ignore",
]


def get_daily_klines(symbol: str) -> pd.DataFrame:
    """Collect all closed daily klines for one symbol in the approved range."""
    pages: list[list[object]] = []
    next_start_ms = START_TIME_MS

    while next_start_ms <= END_TIME_MS:
        payload = request_json(
            KLINES_PATH,
            {
                "symbol": symbol,
                "interval": KLINE_INTERVAL,
                "startTime": next_start_ms,
                "endTime": END_TIME_MS,
                "limit": KLINE_LIMIT,
                "timeZone": 0,
            },
        )
        if not isinstance(payload, list):
            raise ValueError(f"Unexpected kline response for {symbol}.")
        if not payload:
            break
        if any(not isinstance(row, list) or len(row) < 12 for row in payload):
            raise ValueError(f"Unexpected kline row for {symbol}.")

        pages.extend(payload)
        last_open_ms = int(payload[-1][0])
        if last_open_ms < next_start_ms:
            raise ValueError(f"Kline pagination did not advance for {symbol}.")
        next_start_ms = last_open_ms + 1
        if len(payload) < KLINE_LIMIT:
            break

    if not pages:
        return pd.DataFrame(columns=KLINE_COLUMNS)

    frame = pd.DataFrame(pages, columns=KLINE_COLUMNS)
    frame["open_time_ms"] = pd.to_numeric(frame["open_time_ms"], errors="raise")
    frame["close_time_ms"] = pd.to_numeric(frame["close_time_ms"], errors="raise")
    frame["number_of_trades"] = pd.to_numeric(
        frame["number_of_trades"], errors="raise", downcast="integer"
    )
    frame = frame.loc[
        (frame["open_time_ms"] >= START_TIME_MS)
        & (frame["close_time_ms"] <= END_TIME_MS)
    ].copy()
    frame["Open Time (UTC)"] = pd.to_datetime(
        frame["open_time_ms"], unit="ms", utc=True
    )
    frame["Close Time (UTC)"] = pd.to_datetime(
        frame["close_time_ms"], unit="ms", utc=True
    )
    return frame.sort_values("open_time_ms").reset_index(drop=True)

In [5]:
def summarize_pair(pair: dict[str, object]) -> dict[str, object]:
    """Collect and summarize daily trade counts for one trading pair."""
    symbol = str(pair["Trading Pair"])
    klines = get_daily_klines(symbol)

    if klines.empty:
        first_open = pd.NaT
        last_close = pd.NaT
        observed_days = 0
        missing_days_within_observed_span = 0
        total_trades = 0
    else:
        duplicate_days = int(klines["open_time_ms"].duplicated().sum())
        if duplicate_days:
            raise ValueError(f"{symbol} returned {duplicate_days} duplicate days.")
        if (klines["number_of_trades"] < 0).any():
            raise ValueError(f"{symbol} returned a negative trade count.")

        first_open = klines["Open Time (UTC)"].min()
        last_close = klines["Close Time (UTC)"].max()
        observed_days = int(klines["Open Time (UTC)"].nunique())
        expected_days = int(
            (
                klines["Open Time (UTC)"].max()
                - klines["Open Time (UTC)"].min()
            ).days
            + 1
        )
        missing_days_within_observed_span = expected_days - observed_days
        total_trades = int(klines["number_of_trades"].sum())

    return {
        **pair,
        "Collection Start (UTC)": START_TIME_UTC,
        "Collection End (UTC)": END_TIME_UTC,
        "First Observed Kline (UTC)": first_open,
        "Last Observed Kline Close (UTC)": last_close,
        "Observed Daily Klines": observed_days,
        "Missing Days Within Observed Span": missing_days_within_observed_span,
        "Total Number of Trades": total_trades,
        "Source URL": f"{BASE_URL}{KLINES_PATH}",
    }


pair_records = pairs.to_dict(orient="records")
summaries: list[dict[str, object]] = []
errors: list[dict[str, str]] = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(summarize_pair, pair): pair for pair in pair_records}
    for completed, future in enumerate(as_completed(futures), start=1):
        pair = futures[future]
        try:
            summaries.append(future.result())
        except Exception as error:
            errors.append(
                {"Trading Pair": str(pair["Trading Pair"]), "Error": str(error)}
            )
        if completed % 25 == 0 or completed == len(pair_records):
            print(f"Completed {completed:,}/{len(pair_records):,} symbols")

if errors:
    display(pd.DataFrame(errors))
    raise RuntimeError(
        f"Collection stopped because {len(errors)} symbol request(s) failed."
    )

trade_counts = (
    pd.DataFrame(summaries)
    .sort_values(["Total Number of Trades", "Trading Pair"], ascending=[False, True])
    .reset_index(drop=True)
)
trade_counts.head(20)

Completed 25/738 symbols
Completed 50/738 symbols
Completed 75/738 symbols
Completed 100/738 symbols
Completed 125/738 symbols
Completed 150/738 symbols
Completed 175/738 symbols
Completed 200/738 symbols
Completed 225/738 symbols
Completed 250/738 symbols
Completed 275/738 symbols
Completed 300/738 symbols
Completed 325/738 symbols
Completed 350/738 symbols
Completed 375/738 symbols
Completed 400/738 symbols
Completed 425/738 symbols
Completed 450/738 symbols
Completed 475/738 symbols
Completed 500/738 symbols
Completed 525/738 symbols
Completed 550/738 symbols
Completed 575/738 symbols
Completed 600/738 symbols
Completed 625/738 symbols
Completed 650/738 symbols
Completed 675/738 symbols
Completed 700/738 symbols
Completed 725/738 symbols
Completed 738/738 symbols


,Trading Pair,Base Asset,Quote Asset,Current Status,Spot Trading Currently Allowed,Collection Start (UTC),Collection End (UTC),First Observed Kline (UTC),Last Observed Kline Close (UTC),Observed Daily Klines,Missing Days Within Observed Span,Total Number of Trades,Source URL
0,BTCUSDT,BTC,USDT,TRADING,True,2017-01-01 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,2017-08-17 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,3303,0,6645741649,https://api.binance.com/api/v3/klines
1,FLOKIUSDT,FLOKI,USDT,TRADING,True,2017-01-01 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,2023-05-05 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,1216,0,5551020934,https://api.binance.com/api/v3/klines
2,ETHUSDT,ETH,USDT,TRADING,True,2017-01-01 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,2017-08-17 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,3303,0,4325173178,https://api.binance.com/api/v3/klines
3,BONKUSDT,BONK,USDT,TRADING,True,2017-01-01 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,2023-12-15 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,992,0,2365373401,https://api.binance.com/api/v3/klines
4,SOLUSDT,SOL,USDT,TRADING,True,2017-01-01 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,2020-08-11 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,2213,0,2044386668,https://api.binance.com/api/v3/klines
5,XRPUSDT,XRP,USDT,TRADING,True,2017-01-01 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,2018-05-04 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,3043,0,1698108549,https://api.binance.com/api/v3/klines
6,SHIBUSDT,SHIB,USDT,TRADING,True,2017-01-01 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,2021-05-10 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,1941,0,1694795625,https://api.binance.com/api/v3/klines
7,DOGEUSDT,DOGE,USDT,TRADING,True,2017-01-01 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,2019-07-05 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,2616,0,1600460209,https://api.binance.com/api/v3/klines
8,BNBUSDT,BNB,USDT,TRADING,True,2017-01-01 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,2017-11-06 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,3222,0,1574115256,https://api.binance.com/api/v3/klines
9,ADAUSDT,ADA,USDT,TRADING,True,2017-01-01 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,2018-04-17 00:00:00+00:00,2026-09-01 23:59:59.999000+00:00,3060,0,778159344,https://api.binance.com/api/v3/klines


In [6]:
validation = pd.DataFrame(
    [
        ("Duplicate trading pairs", int(trade_counts["Trading Pair"].duplicated().sum()), 0),
        ("Unexpected quote assets", int((trade_counts["Quote Asset"] != QUOTE_ASSET).sum()), 0),
        ("Negative total trade counts", int((trade_counts["Total Number of Trades"] < 0).sum()), 0),
        ("Negative missing-day counts", int((trade_counts["Missing Days Within Observed Span"] < 0).sum()), 0),
        ("Symbols lost during collection", len(pairs) - len(trade_counts), 0),
    ],
    columns=["Check", "Actual", "Expected"],
)
validation["Status"] = validation.apply(
    lambda row: "OK" if row["Actual"] == row["Expected"] else "FAIL",
    axis=1,
)
if not (validation["Status"] == "OK").all():
    display(validation)
    raise ValueError("Validation failed; the workbook will not be generated.")
validation

,Check,Actual,Expected,Status
0,Duplicate trading pairs,0,0,OK
1,Unexpected quote assets,0,0,OK
2,Negative total trade counts,0,0,OK
3,Negative missing-day counts,0,0,OK
4,Symbols lost during collection,0,0,OK


In [7]:
methodology = pd.DataFrame(
    [
        ("Collection timestamp (UTC)", COLLECTION_TIME_UTC.isoformat()),
        ("Market", "Binance Spot"),
        ("Universe definition", "All symbols in the current exchangeInfo response with quoteAsset=USDT; no status filter."),
        ("Historical-universe limitation", "Symbols removed from the current exchangeInfo response cannot be discovered by this procedure."),
        ("Start (UTC)", START_TIME_UTC.isoformat()),
        ("End (UTC)", END_TIME_UTC.isoformat()),
        ("Kline interval", KLINE_INTERVAL),
        ("Activity measure", "Sum of kline field 8 (number_of_trades) per trading symbol."),
        ("Pair specificity", "Counts belong to the full symbol and depend on both base and quote assets."),
        ("Timezone", "UTC (timeZone=0)"),
        ("Exchange information endpoint", f"{BASE_URL}{EXCHANGE_INFO_PATH}"),
        ("Kline endpoint", f"{BASE_URL}{KLINES_PATH}"),
    ],
    columns=["Field", "Value"],
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
export_data = trade_counts.copy()
datetime_columns = [
    "Collection Start (UTC)",
    "Collection End (UTC)",
    "First Observed Kline (UTC)",
    "Last Observed Kline Close (UTC)",
]
for column_name in datetime_columns:
    export_data[column_name] = export_data[column_name].dt.tz_localize(None)

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    export_data.to_excel(writer, sheet_name="Pair Trade Counts", index=False)
    validation.to_excel(writer, sheet_name="Validation", index=False)
    methodology.to_excel(writer, sheet_name="Methodology", index=False)

print(f"Workbook created: {OUTPUT_PATH}")

Workbook created: /Users/marcelapsilverio/projetos/tcc_crypto/data/exploration/binance_spot_usdt_quote_pair_trade_counts.xlsx


## Interpretation

`Total Number of Trades` is a count of executions reported by Binance for each full Spot symbol during the requested period. It is not the quantity of base units traded and it is not an asset-level count. To obtain an asset-level figure across quote currencies, one would need an explicitly defined aggregation across multiple symbols; such a sum would answer a different question.

`Missing Days Within Observed Span` is a coverage diagnostic. Review any positive value before using the totals for pair selection. Days before the first observed kline and after the last observed kline are not labeled as missing because they may precede listing or follow delisting.